# 🚕 NYC Taxi Trip Duration — End-to-End Data Science

**Framework:** CRISP-DM  
**Task:** Regression  
**Metric:** RMSLE  
**Deployment:** Streamlit + Joblib

The notebook runs in:

1. **Official mode** when `data/train.csv` and `data/test.csv` exist.
2. **Smoke-test mode** using included synthetic schema-compatible sample files.

> Synthetic sample scores are not Kaggle results.


## CRISP-DM 1 — Business Understanding

### Objective
Predict total taxi trip duration in seconds using information available at or near pickup time.

### Business value
Useful for ETA communication, passenger planning, dispatch support, fleet analytics, and service monitoring.

### Success metric
Kaggle evaluates with **RMSLE**:

\[
RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\log(\hat y_i+1)-\log(y_i+1))^2}
\]

Lower is better.

### Leakage constraint
`dropoff_datetime` is available only after the trip ends. It must never be used as a predictor.


In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from xgboost import XGBRegressor

SEED = 42

CWD = Path.cwd()
if (CWD / "src").exists():
    ROOT = CWD
elif (CWD.parent / "src").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError("Run from the repository root or notebooks/ directory.")

sys.path.insert(0, str(ROOT))
from src.features import build_features, FEATURE_COLUMNS
from src.modeling import rmsle

DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "outputs"

official_train = DATA_DIR / "train.csv"
official_test = DATA_DIR / "test.csv"

if official_train.exists() and official_test.exists():
    TRAIN_PATH, TEST_PATH = official_train, official_test
    USING_SAMPLE = False
else:
    TRAIN_PATH = DATA_DIR / "sample_train.csv"
    TEST_PATH = DATA_DIR / "sample_test.csv"
    USING_SAMPLE = True

MAX_TRAIN_ROWS = None if USING_SAMPLE else 300_000

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print("Mode :", "SYNTHETIC SMOKE TEST" if USING_SAMPLE else "OFFICIAL KAGGLE DATA")
print("Row cap:", MAX_TRAIN_ROWS)


## CRISP-DM 2 — Data Understanding

Inspect shape, schema, missingness, duplicates, target skew, passengers, temporal coverage, coordinate ranges, and suspicious records.


In [ ]:
train = pd.read_csv(TRAIN_PATH, nrows=MAX_TRAIN_ROWS)
test = pd.read_csv(TEST_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
display(train.head())


In [ ]:
quality = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean()*100).round(3),
    "nunique": train.nunique(dropna=False)
})
display(quality)
print("Duplicate rows:", train.duplicated().sum())


In [ ]:
display(train["trip_duration"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

plt.figure(figsize=(10,6))
plt.hist(np.log1p(train["trip_duration"].clip(lower=0)), bins=60)
plt.xlabel("log1p(trip_duration)")
plt.ylabel("Trips")
plt.title("Target Distribution in Log Space")
plt.tight_layout()
plt.show()


In [ ]:
pickup_dt = pd.to_datetime(train["pickup_datetime"], errors="coerce")
print("Pickup range:", pickup_dt.min(), "→", pickup_dt.max())

geo_cols = [
    "pickup_longitude","pickup_latitude",
    "dropoff_longitude","dropoff_latitude"
]
display(train[geo_cols].describe())
display(train["passenger_count"].value_counts(dropna=False).sort_index().head(15))


### Training-only cleaning

Remove clearly unusable target values, impossible passenger counts, and coordinates far outside a broad NYC region.

The test set is not filtered because Kaggle requires one prediction per test ID.


In [ ]:
clean = train.copy()
before = len(clean)

clean = clean[clean["trip_duration"].between(30, 6*60*60)]
clean = clean[clean["passenger_count"].between(1,6)]

clean = clean[
    clean["pickup_latitude"].between(40.45,41.05)
    & clean["dropoff_latitude"].between(40.45,41.05)
    & clean["pickup_longitude"].between(-74.30,-73.55)
    & clean["dropoff_longitude"].between(-74.30,-73.55)
].copy()

print(f"Before: {before:,}")
print(f"After : {len(clean):,}")
print(f"Removed: {before-len(clean):,} ({(before-len(clean))/before:.2%})")


## CRISP-DM 3 — Data Preparation

### Temporal features
Month, day, weekday, hour, minute, ISO week, weekend, rush hour.

### Geospatial features
Haversine distance, Manhattan-style distance, bearing, route center.

### Other
Vendor, passenger count, store-and-forward encoding.

`dropoff_datetime` is deliberately excluded.


In [ ]:
X = build_features(clean)
y = clean["trip_duration"].to_numpy(dtype=float)
y_log = np.log1p(y)

print("Feature matrix:", X.shape)
display(X.head())


In [ ]:
X_train, X_valid, y_train_log, y_valid_log, y_train_raw, y_valid_raw = train_test_split(
    X, y_log, y, test_size=.20, random_state=SEED
)
print(X_train.shape, X_valid.shape)


## CRISP-DM 4 — Modeling

Models train on `log1p(trip_duration)`, aligning naturally with the log-based competition metric.


In [ ]:
models = {
    "Dummy Median": DummyRegressor(strategy="median"),
    "Random Forest": RandomForestRegressor(
        n_estimators=220, max_depth=18, min_samples_leaf=3,
        n_jobs=-1, random_state=SEED
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=260, learning_rate=.06, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=SEED
    ),
    "XGBoost": XGBRegressor(
        n_estimators=550, max_depth=8, learning_rate=.05,
        subsample=.90, colsample_bytree=.90, min_child_weight=3,
        reg_lambda=1.0, objective="reg:squarederror", eval_metric="rmse",
        n_jobs=-1, random_state=SEED
    )
}

comparison, fitted_models = [], {}

for name, model in models.items():
    model.fit(X_train, y_train_log)
    pred = np.clip(np.expm1(model.predict(X_valid)), 0, None)
    score = rmsle(y_valid_raw, pred)
    comparison.append({"model":name, "validation_RMSLE":score})
    fitted_models[name] = model
    print(f"{name:22s} RMSLE = {score:.5f}")

comparison_df = pd.DataFrame(comparison).sort_values("validation_RMSLE").reset_index(drop=True)
display(comparison_df)


In [ ]:
plot_df = comparison_df.sort_values("validation_RMSLE")
plt.figure(figsize=(10,6))
plt.barh(plot_df["model"], plot_df["validation_RMSLE"])
plt.xlabel("Validation RMSLE (lower is better)")
plt.title("Model Comparison")
for i,v in enumerate(plot_df["validation_RMSLE"]):
    plt.text(v, i, f"  {v:.4f}", va="center")
plt.tight_layout()
plt.show()


## CRISP-DM 5 — Evaluation

Use cross-validation to test whether the strongest practical model is stable beyond one holdout split.


In [ ]:
best_name = comparison_df.loc[~comparison_df["model"].eq("Dummy Median"), "model"].iloc[0]
best_model = fitted_models[best_name]
print("Best holdout candidate:", best_name)

cv_n = min(len(X), 120_000 if not USING_SAMPLE else len(X))
cv_idx = np.random.default_rng(SEED).choice(len(X), size=cv_n, replace=False)
X_cv = X.iloc[cv_idx]
y_cv_log = y_log[cv_idx]

cv = KFold(n_splits=3, shuffle=True, random_state=SEED)
cv_scores = -cross_val_score(
    best_model, X_cv, y_cv_log,
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=1
)

print("Fold scores:", np.round(cv_scores,5))
print("Mean:", cv_scores.mean().round(5))
print("Std :", cv_scores.std().round(5))


### Light XGBoost tuning

The search is intentionally small to balance improvement against runtime and overfitting risk.


In [ ]:
tune_n = min(len(X_train), 100_000 if not USING_SAMPLE else len(X_train))
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_train), size=tune_n, replace=False)

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1,
    random_state=SEED
)

param_dist = {
    "n_estimators":[350,500,700],
    "max_depth":[5,7,9],
    "learning_rate":[.03,.05,.08],
    "subsample":[.8,.9,1.0],
    "colsample_bytree":[.8,.9,1.0],
    "min_child_weight":[1,3,6],
    "reg_lambda":[1.0,3.0,6.0]
}

search = RandomizedSearchCV(
    xgb_base, param_dist, n_iter=6,
    scoring="neg_root_mean_squared_error",
    cv=3, random_state=SEED, n_jobs=1, verbose=1
)
search.fit(X_train.iloc[idx], y_train_log[idx])

print("Best params:", search.best_params_)
print("Best CV log-RMSE:", -search.best_score_)


In [ ]:
tuned_model = search.best_estimator_
tuned_pred = np.clip(np.expm1(tuned_model.predict(X_valid)), 0, None)
tuned_rmsle = rmsle(y_valid_raw, tuned_pred)

original_xgb_rmsle = float(
    comparison_df.loc[comparison_df["model"]=="XGBoost","validation_RMSLE"].iloc[0]
)

print(f"Original XGBoost RMSLE: {original_xgb_rmsle:.5f}")
print(f"Tuned XGBoost RMSLE   : {tuned_rmsle:.5f}")
print(f"Improvement           : {original_xgb_rmsle-tuned_rmsle:+.5f}")

if tuned_rmsle < original_xgb_rmsle:
    print("Recommendation: use tuned XGBoost.")
else:
    print("Recommendation: keep original XGBoost.")


In [ ]:
candidate = tuned_model if tuned_rmsle < original_xgb_rmsle else fitted_models["XGBoost"]
candidate_name = "Tuned XGBoost" if tuned_rmsle < original_xgb_rmsle else "Original XGBoost"

pred_valid = np.clip(np.expm1(candidate.predict(X_valid)), 0, None)
residual_log = np.log1p(y_valid_raw) - np.log1p(pred_valid)

print(pd.Series(residual_log).describe())

plt.figure(figsize=(10,6))
plt.hist(residual_log, bins=60)
plt.xlabel("log1p(actual) - log1p(predicted)")
plt.ylabel("Trips")
plt.title(f"Residual Distribution — {candidate_name}")
plt.tight_layout()
plt.show()


In [ ]:
if hasattr(candidate, "feature_importances_"):
    importance = pd.Series(candidate.feature_importances_, index=FEATURE_COLUMNS).sort_values().tail(15)
    display(importance.sort_values(ascending=False).to_frame("importance"))
    plt.figure(figsize=(10,7))
    plt.barh(importance.index, importance.values)
    plt.xlabel("Feature importance")
    plt.title(f"Top Features — {candidate_name}")
    plt.tight_layout()
    plt.show()


### Skeptical review checklist

- Confirm no future/post-trip information entered the feature set.
- Compare holdout and CV behavior for overfitting.
- Document cleaning thresholds.
- Consider time-aware validation for future deployment.
- Check geographic shift and unusual routes.
- Remember that 2016 travel behavior can drift.
- Keep evaluation aligned to RMSLE.


## Final model serialization

The saved artifact stores the trained model, feature column order, validation metadata, target transform, row count, and random seed.


In [ ]:
final_estimator = search.best_estimator_ if tuned_rmsle < original_xgb_rmsle else models["XGBoost"]
final_estimator.fit(X, y_log)

final_bundle = {
    "model": final_estimator,
    "feature_columns": FEATURE_COLUMNS,
    "metadata": {
        "model_name": "Tuned XGBoost" if tuned_rmsle < original_xgb_rmsle else "XGBoost",
        "validation_rmsle": float(min(tuned_rmsle, original_xgb_rmsle)),
        "trained_on": "synthetic schema-compatible sample data" if USING_SAMPLE else "official Kaggle train.csv",
        "target_transform": "log1p",
        "rows_used": int(len(X)),
        "random_seed": SEED
    }
}

model_path = MODEL_DIR / "taxi_duration_model.joblib"
joblib.dump(final_bundle, model_path)
print("Saved:", model_path)


## Kaggle submission generation

Build the same features for the test set, predict in log space, transform back to seconds, and write `id,trip_duration`.


In [ ]:
X_test = build_features(test)
test_pred = np.maximum(np.expm1(final_estimator.predict(X_test)), 1.0)

submission = pd.DataFrame({
    "id": test["id"],
    "trip_duration": test_pred
})

submission_path = OUTPUT_DIR / ("sample_submission_generated.csv" if USING_SAMPLE else "submission.csv")
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())
print("Shape:", submission.shape)


## CRISP-DM 6 — Deployment

Run the Streamlit app:

```bash
streamlit run app.py
```

The app imports the same `src/features.py` used during training, reducing training-serving skew.

### Production recommendations
- schema and coordinate validation
- latency/error logging
- input and concept drift monitoring
- scheduled retraining
- live route/traffic features for real ETA use
- prediction intervals instead of only point estimates


# Final conclusion

This repository implements:

**Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment**

Key design choices:

1. prevent leakage by excluding `dropoff_datetime`;
2. align training with RMSLE using a log target;
3. engineer temporal and geospatial route features;
4. compare complex models against a naive baseline;
5. use CV and restrained tuning;
6. reuse identical feature code in the deployed app.

When official Kaggle data is present, rerun all cells and report only official-data results.
